In [1]:
import sys
from pathlib import Path
sys.path[:0] = [str(Path.cwd().parent)]

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from self_consistency_k import bdg_sc_full_k

In [2]:
corr_strings = [
    "F_onsite",
    "F_swave",
    "F_dwave",
    "F_px",
    "F_py",
    "Fuu_px",
    "Fuu_py",
    "Fdd_px",
    "Fdd_py",
]

In [3]:
def stable_config(out, atol=1e-6, rtol=0.1):
    stable = {}
    F_max = out.corr[np.argmax(np.abs(out.corr))]
    if np.abs(F_max) < atol:
        return stable
    for i, c in enumerate(out[1:-3]):
        if np.abs(c) > atol and np.abs(c) > rtol * np.abs(F_max):
            stable[corr_strings[i]] = c
    return stable

In [4]:
def same_stable_dict(d1, d2, atol=1e-6, rtol=1e-3):
    if d1.keys() != d2.keys():
        return False

    for k in d1:
        if not np.isclose(d1[k], d2[k], atol=atol, rtol=rtol):
            return False

    return True

In [5]:
initial_seeds = np.array([
    [0,0,0,0],  # normal state
    [0.1, -0.1, 0, 0],  # px 
    [0, 0, 0.1, -0.1],  # py
    [0.1, -0.1, 0.1, -0.1],  # px + py
    [0.1, -0.1, 0.1j, -0.1j],  # px + i*py
    # [0.1, 0.1, -0.1, -0.1],  # d-wave
    # [0.1+0.1, 0.1-0.1, -0.1, -0.1],  # d-wave + px
    # [0.1, 0.1, -0.1+0.1, -0.1-0.1],  # d-wave + py
    [0.1, 0.1, 0.1, 0.1],  # s-wave
    [0.1+0.1, 0.1-0.1, 0.1, 0.1],  # s-wave + px
    [0.1, 0.1, 0.1+0.1, 0.1-0.1] # s-wave + py
], dtype=np.complex128)
seed_strings = [
    "normal state",
    "px", "py", "px+py", "px+i*py",
    # "d-wave", "d-wave+px", "d-wave+py",
    "s-wave", "s-wave+px", "s-wave+py"
]

In [18]:
mu_arr = np.linspace(1.8, 4.0, 1)
T_arr = np.linspace(0.001, 0.2, 1)

mu_arr = [2.215, 2.225]
T_arr = [0.001]
free_tol = 0.1
print(mu_arr)
print(T_arr)

[2.215, 2.225]
[0.001]


In [19]:
t=1
V_prime=1.5
V = 1.5

h = np.array([0, 0, 0])
Nx, Ny = 50, 50

atol = 1e-8
rtol = 1e-4
maxiter=1000

In [20]:
print_output = True
records = []

for mu in mu_arr:
    for T in T_arr:
        best_free = np.inf
        configs = []
        print("====================================")
        print(f"mu={mu:.2f}, T={T:.4f}")
        print("====================================")
        for seed, seed_str in zip(initial_seeds, seed_strings):
            print(f"  Seed: {seed_str}")

            out = bdg_sc_full_k(
                t, mu, temperature=T,
                V=V, Nx=Nx, Ny=Ny,
                atol=atol, rtol=rtol,
                maxiter=maxiter,
                F_init=seed
            )
            print(f"F_onsite: {out.F_onsite}")
            print(f"F_swave: {out.F_swave}")
            print(f"F_px: {out.F_px}")
            print(f"F_py: {out.F_py}")
            print(f"Free energy: {out.free_energy}")

            stable = stable_config(out, atol=atol)

            print("------------------------------------------")
            if (out.free_energy < best_free and
                    np.abs(out.free_energy - best_free) > free_tol):

                best_free = out.free_energy
                configs = [{
                    "stable": stable,
                    "free": out.free_energy,
                }]

            elif np.abs(out.free_energy - best_free) <= free_tol:
                if len(stable) != 0 and not any(
                    same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
                    for c in configs
                ):
                    configs.append({
                        "stable": stable,
                        "free": out.free_energy,
                    })

            print(configs)
            print("------------------------------------------")

        records.append({
            "mu": mu,
            "T": T,
            "best_free": best_free,
            "configs": configs
        })

    # Create DataFrame
df = pd.DataFrame(records)


mu=2.21, T=0.0010
  Seed: normal state
F_onsite: 0j
F_swave: 0j
F_px: 0j
F_py: 0j
Free energy: -6227.113324422569
------------------------------------------
[{'stable': {}, 'free': np.float64(-6227.113324422569)}]
------------------------------------------
  Seed: px
F_onsite: -8.138874892370555e-18j
F_swave: 4.8411374006931485e-18j
F_px: (0.008509118067968623+0j)
F_py: (3.7489717950144736e-17+0j)
Free energy: -6227.6776829904165
------------------------------------------
[{'stable': {'F_px': np.complex128(0.008509118067968623+0j)}, 'free': np.float64(-6227.6776829904165)}]
------------------------------------------
  Seed: py
F_onsite: -7.228155655394572e-18j
F_swave: 4.8006121681229506e-18j
F_px: (3.5727313702149454e-17+0j)
F_py: (0.008509118067968589+0j)
Free energy: -6227.677682990409
------------------------------------------
[{'stable': {'F_px': np.complex128(0.008509118067968623+0j)}, 'free': np.float64(-6227.6776829904165)}, {'stable': {'F_py': np.complex128(0.00850911806796858

In [22]:
print(df[0:100])

      mu      T    best_free  \
0  2.215  0.001 -6227.839766   
1  2.225  0.001 -6245.194839   

                                             configs  
0  [{'stable': {'F_px': (0.006823617567259511-3.9...  
1  [{'stable': {'F_onsite': (-0.01278073776364674...  


In [23]:
def flatten_df(df):
    df_flat = df.copy()

    # Make sure configs is always a list before exploding
    df_flat["configs"] = df_flat["configs"].apply(
        lambda x: x if isinstance(x, list)
        else [x] if isinstance(x, dict)
        else []
    )

    df_flat = df_flat.explode("configs").reset_index(drop=True)

    # Split out the free energy and stable dict
    df_flat["free"] = df_flat["configs"].apply(
        lambda x: x.get("free", 0) if isinstance(x, dict) else 0
    )
    df_flat["stable"] = df_flat["configs"].apply(
        lambda x: x.get("stable", {}) if isinstance(x, dict) else {}
    )

    # Expand the stable dictionaries into columns
    stable_df = pd.DataFrame(df_flat["stable"].tolist())

    # Ensure all desired columns exist
    stable_df = stable_df.reindex(columns=corr_strings, fill_value=0)

    # Combine everything
    df_flat = pd.concat(
        [
            df_flat.drop(columns=["configs", "stable"]),
            stable_df
        ],
        axis=1
    )

    df_flat.fillna(0, inplace=True)
    return df_flat


In [24]:
df_flat = flatten_df(df)

df_flat.iloc[:]

,mu,T,best_free,free,F_onsite,F_swave,F_dwave,F_px,F_py,Fuu_px,Fuu_py,Fdd_px,Fdd_py
0,2.215,0.001,-6227.839766,-6227.839766,0.000000+0.000000j,0.000000+0.000000j,0,0.006824-0.000000j,0.000000+0.006824j,0,0,0,0
1,2.215,0.001,-6227.839766,-6227.916080,-0.010510+0.000000j,0.007155+0.000000j,0,0.000000+0.000000j,0.000000+0.000000j,0,0,0,0
2,2.215,0.001,-6227.839766,-6227.839766,0.000000+0.000000j,0.000000+0.000000j,0,0.006824+0.000000j,0.000000-0.006824j,0,0,0,0
3,2.215,0.001,-6227.839766,-6227.839766,0.000000+0.000000j,0.000000+0.000000j,0,-0.000000-0.006824j,0.006824-0.000000j,0,0,0,0
4,2.225,0.001,-6245.194839,-6245.194839,-0.012781+0.000000j,0.008730+0.000000j,0,0.000000+0.000000j,0.000000+0.000000j,0,0,0,0


In [25]:
# df_flat.to_parquet("results_flat.parquet", index=False)
df_flat.to_json("results.json")